# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
#loading the data
!pip install -q duckdb huggingface_hub

import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

base = "hf://datasets/FlyRank/internship-warehouse"
df_march_agg = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS gsc_impressions,
           SUM(gsc_clicks) AS gsc_clicks,
           SUM(gsc_sum_position) AS gsc_sum_position,
           SUM(ga4_sessions) AS ga4_sessions,
           SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    GROUP BY client_hash_id, content_hash_id
""").df()

df_april_agg = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_clicks) AS gsc_clicks_apr
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-04/data_0.parquet')
    GROUP BY client_hash_id, content_hash_id
""").df()

print(f"March pages: {len(df_march_agg)}, April pages: {len(df_april_agg)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March pages: 331437, April pages: 362172


In [4]:
df_april_agg.head()

,client_hash_id,content_hash_id,gsc_clicks_apr
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,0.0
1,client_62f4a7e64f5e0096,content_13a8105125458098,0.0
2,client_62f4a7e64f5e0096,content_6a887d56ab6c8362,0.0
3,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,0.0
4,client_62f4a7e64f5e0096,content_86ab16840c4e0e1a,1.0


In [2]:
#last week's rule rewritten
import pandas as pd
import numpy as np

agg_df = df_march_agg.copy()

agg_df['gsc_avg_position'] = agg_df['gsc_sum_position'] / agg_df['gsc_impressions']
agg_df.loc[agg_df['gsc_sum_position'] == 0, 'gsc_avg_position'] = pd.NA

def position_bucket(pos):
    if pd.isna(pos):
        return 'no_position_data'
    elif pos <= 3:
        return '1-3 (top)'
    elif pos <= 10:
        return '4-10'
    elif pos <= 20:
        return '11-20'
    else:
        return '21+'

signal_df = agg_df[agg_df['gsc_impressions'] > 0].copy()
signal_df['ctr'] = signal_df['gsc_clicks'] / signal_df['gsc_impressions']
signal_df['position_bucket'] = signal_df['gsc_avg_position'].apply(position_bucket)
position_avg_ctr = signal_df.groupby('position_bucket')['ctr'].mean().to_dict()

rule_df = agg_df.copy()
rule_df['ctr'] = rule_df['gsc_clicks'] / rule_df['gsc_impressions']
rule_df['engagement_rate'] = rule_df['ga4_engaged_sessions'] / rule_df['ga4_sessions']
rule_df['position_bucket'] = rule_df['gsc_avg_position'].apply(position_bucket)
rule_df['peer_avg_ctr'] = rule_df['position_bucket'].map(position_avg_ctr)

eligible = (rule_df['gsc_impressions'] >= 50) | (rule_df['ga4_sessions'] >= 10)
rule_df = rule_df[eligible].copy()

conditions = [
    (rule_df['gsc_impressions'] >= 50) & rule_df['ctr'].notna() & rule_df['peer_avg_ctr'].notna()
        & (rule_df['ctr'] < 0.5 * rule_df['peer_avg_ctr']),
    (rule_df['ga4_sessions'] >= 10) & rule_df['engagement_rate'].notna()
        & (rule_df['engagement_rate'] < 0.10),
]
rule_df['action'] = np.select(conditions, ['snippet_fix', 'content_fix'], default='monitor')
rule_df['reason_code'] = np.select(conditions,
    ['CTR_below_half_position_peers', 'engagement_below_10pct_reliable'], default='no_flag_triggered')
rule_df['rule_score'] = np.select(conditions,
    [(rule_df['peer_avg_ctr'] - rule_df['ctr']) * rule_df['gsc_impressions'],
     rule_df['ga4_sessions'] * (0.10 - rule_df['engagement_rate']) * 10], default=0)

print(f"March-eligible population: {len(rule_df)} pages")
print(rule_df['action'].value_counts())

March-eligible population: 116512 pages
action
snippet_fix    75663
monitor        28755
content_fix    12094
Name: count, dtype: int64


 **Rule**: decline_label = 1 if April clicks < 80% of March clicks

 Only counted if: (a) the page is still tracked in April at all, and

(b) it had at least 5 clicks in March (so the ratio isn't noise off tiny counts)

In [3]:
# April outcome
MIN_MARCH_CLICKS_FOR_LABEL = 5
DECLINE_THRESHOLD = 0.8

merged = rule_df.merge(df_april_agg, on=['client_hash_id', 'content_hash_id'], how='left')
tracked_in_april = merged['gsc_clicks_apr'].notna()
print(f"March pages with no April tracking at all (excluded, can't score a fair outcome): {(~tracked_in_april).sum()}")

labelable = merged[tracked_in_april & (merged['gsc_clicks'] >= MIN_MARCH_CLICKS_FOR_LABEL)].copy()
labelable['decline_label'] = (labelable['gsc_clicks_apr'] < DECLINE_THRESHOLD * labelable['gsc_clicks']).astype(int)

print(f"Excluded for < {MIN_MARCH_CLICKS_FOR_LABEL} March clicks: {(tracked_in_april & (merged['gsc_clicks'] < MIN_MARCH_CLICKS_FOR_LABEL)).sum()}")
print(f"Final labelable population: {len(labelable)} pages")
print(f"Base rate (decline_label == 1): {labelable['decline_label'].mean():.3f}")

March pages with no April tracking at all (excluded, can't score a fair outcome): 0
Excluded for < 5 March clicks: 87707
Final labelable population: 28805 pages
Base rate (decline_label == 1): 0.545


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.